In [2]:
##import needed tools
import re
from pathlib import Path
from collections import Counter

import nltk
import spacy
import pandas as pd

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

In [4]:
#download and load additional components
nltk.download("punkt")
nltk.download("stopwords")
nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sabri\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sabri\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
# -------------------------
# FILE PATHS
# -------------------------

# defines paths to each file (stored in "data")
fiction_path = Path("./data/fiction.text")
cookbook_path = Path("./data/cookbook.text")
textbook_path = Path("./data/textbook.text")

# stores file paths in a dictionary for easy iteration
files = {
    "Fiction": fiction_path,
    "Cookbook": cookbook_path,
    "Textbook": textbook_path
}

In [6]:
# -------------------------
# FILE READING
# -------------------------

# opens and reads a text file. returns full text as a string.
def read_text(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return f.read()

In [7]:
# -------------------------
# TEXT CLEANING
# -------------------------

# cleans raw text for analysis and returns cleaned text.
def clean_text(text):
    
    # removes header
    text = re.sub(r"\*\*\* START OF THE PROJECT GUTENBERG EBOOK .*? \*\*\*", "", text, flags=re.DOTALL)
    
    # removes footer
    text = re.sub(r"\*\*\* END OF THE PROJECT GUTENBERG EBOOK .*? \*\*\*", "", text, flags=re.DOTALL)
    
    # converts all text to lowercase
    text = text.lower()

    # removes extra spaces and line breaks
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [8]:
# -------------------------
# TOKENIZATION
# -------------------------

# tokenizes text into words, removing numbers and symbols, leaving only letters 
def get_word_tokens(text):
    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token.isalpha()]
    return tokens

In [9]:
# -------------------------
# LEXICAL DIVERSITY
# -------------------------

# computes lexical diversity by calculating the amount of unique tokens by the total number of tokens
def lexical_diversity(tokens):
    if len(tokens) == 0:
        return 0
    return len(set(tokens)) / len(tokens)

In [10]:
# -------------------------
# SENTENCE LENGTH ANALYSIS
# -------------------------

# calculates average number of tokens per sentence
def average_sentence_length(text):
    sentences = sent_tokenize(text)
    if len(sentences) == 0:
        return 0
    sentence_lengths = []

    # tokenizes words
    for sentence in sentences:
        words = [word for word in word_tokenize(sentence) if word.isalpha()]
        if words:
            sentence_lengths.append(len(words))

    # avoids diving by 0
    if len(sentence_lengths) == 0:
        return 0

    # computes average sentence length
    return sum(sentence_lengths) / len(sentence_lengths)

In [11]:
# -------------------------
# TOP WORD FREQUENCY
# -------------------------

# counts most frequent words, excluding stopwords
def top_words(tokens, top_n=10):
    stop_words = set(stopwords.words("english"))
    filtered = [token for token in tokens if token not in stop_words]
    return Counter(filtered).most_common(top_n)

In [12]:
# -------------------------
# NAMED ENTITY ANALYSIS
# -------------------------

# counts named entities
def named_entity_counts(text):
    doc = nlp(text)

    #extracts only entity labels rather than the text itself
    entities = [ent.label_ for ent in doc.ents]
    total_entities = len(entities)
    label_counts = Counter(entities)

    return total_entities, label_counts

In [13]:
# -------------------------
# MAIN ANALYSIS FUNCTION
# -------------------------

# reads and preprocesses text
def analyze_text(label, filepath):
    raw_text = read_text(filepath)
    cleaned = clean_text(raw_text)
    tokens = get_word_tokens(cleaned)

    # calculates lexical stats
    total_tokens = len(tokens) #total tokens
    total_types = len(set(tokens)) #total unique tokens
    ttr = lexical_diversity(tokens) #lexical diversity
    avg_sent_len = average_sentence_length(raw_text) #sentence length
    common_words = top_words(tokens, top_n=10) #top words
    total_ents, ent_labels = named_entity_counts(raw_text) #named entitities

    # returns all calculations in structured formatting
    return {
        "Text": label,
        "Total Words": total_tokens,
        "Unique Words": total_types,
        "Lexical Diversity": round(ttr, 4),
        "Average Sentence Length": round(avg_sent_len, 2),
        "Total Named Entities": total_ents,
        "Top 10 Words": common_words,
        "Named Entity Labels": dict(ent_labels)
    }

# runs analysis on all files
results = []
for label, filepath in files.items():
    result = analyze_text(label, filepath)
    results.append(result)


In [14]:
# -------------------------
# SUMMARY AND OUTPUT
# -------------------------

# converts results into pandas dataframe
summary_df = pd.DataFrame([{
    "Text": r["Text"],
    "Total Words": r["Total Words"],
    "Unique Words": r["Unique Words"],
    "Lexical Diversity": r["Lexical Diversity"],
    "Average Sentence Length": r["Average Sentence Length"],
    "Total Named Entities": r["Total Named Entities"]
} for r in results])

# displays structured output table
print("SUMMARY TABLE")
display(summary_df)

# goes through each texts to display detailed results 
for r in results:
    print("=" * 60)
    print(f"TEXT: {r['Text']}")
    print(f"Total Words: {r['Total Words']}")
    print(f"Unique Words: {r['Unique Words']}")
    print(f"Lexical Diversity: {r['Lexical Diversity']}")
    print(f"Average Sentence Length: {r['Average Sentence Length']}")
    print(f"Total Named Entities: {r['Total Named Entities']}")

    #displays top meaningful words
    print("\nTop 10 Words:")
    for word, freq in r["Top 10 Words"]:
        print(f"  {word}: {freq}")

    #displays named entity distribution
    print("\nNamed Entity Labels:")
    for label, count in r["Named Entity Labels"].items():
        print(f"  {label}: {count}")
    print()

SUMMARY TABLE


,Text,Total Words,Unique Words,Lexical Diversity,Average Sentence Length,Total Named Entities
0,Fiction,74602,6887,0.0923,24.27,1692
1,Cookbook,4189,812,0.1938,15.95,415
2,Textbook,139116,6491,0.0467,20.52,10144


TEXT: Fiction
Total Words: 74602
Unique Words: 6887
Lexical Diversity: 0.0923
Average Sentence Length: 24.27
Total Named Entities: 1692

Top 10 Words:
  one: 202
  could: 196
  would: 183
  yet: 152
  man: 136
  father: 133
  upon: 126
  life: 112
  every: 109
  first: 108

Named Entity Labels:
  PERSON: 486
  LOC: 27
  LAW: 42
  GPE: 214
  DATE: 264
  CARDINAL: 179
  ORDINAL: 110
  TIME: 153
  LANGUAGE: 10
  NORP: 67
  QUANTITY: 9
  ORG: 120
  WORK_OF_ART: 5
  FAC: 4
  PRODUCT: 2

TEXT: Cookbook
Total Words: 4189
Unique Words: 812
Lexical Diversity: 0.1938
Average Sentence Length: 15.95
Total Named Entities: 415

Top 10 Words:
  cake: 109
  teaspoon: 68
  frosting: 59
  cup: 55
  beat: 37
  top: 36
  add: 35
  spry: 32
  cups: 31
  sugar: 30

Named Entity Labels:
  CARDINAL: 243
  ORG: 29
  PERSON: 62
  QUANTITY: 15
  LOC: 18
  FAC: 2
  DATE: 11
  PRODUCT: 2
  WORK_OF_ART: 3
  TIME: 8
  NORP: 9
  GPE: 8
  ORDINAL: 4
  MONEY: 1

TEXT: Textbook
Total Words: 139116
Unique Words: 6491
Lex